In [1]:
!git clone https://github.com/iloveunk1310/LAMDA-PILOT.git
%cd LAMDA-PILOT

Cloning into 'LAMDA-PILOT'...
remote: Enumerating objects: 627, done.
remote: Counting objects: 100% (472/472), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 627 (delta 358), reused 319 (delta 257), pack-reused 155 (from 1)
Receiving objects: 100% (627/627), 7.41 MiB | 20.38 MiB/s, done.
Resolving deltas: 100% (388/388), done.
/content/LAMDA-PILOT


In [2]:
!pip install -q timm==0.6.12 easydict wandb
# Tối ưu num_workers từ 8 xuống 2 để khớp với 2 vCPU của Colab
!sed -i 's/num_workers = 8/num_workers = 2/g' models/*.py


In [3]:
!python -c "import site, glob, os; [open(p, 'w').write(open(p).read().replace('from dataclasses import dataclass, replace', 'from dataclasses import dataclass, replace, field').replace('MaxxVitConvCfg()', 'field(default_factory=MaxxVitConvCfg)').replace('MaxxVitTransformerCfg()', 'field(default_factory=MaxxVitTransformerCfg)')) for p in glob.glob(os.path.join(site.getsitepackages()[0], 'timm', 'models', 'maxxvit.py'))]"
# 3. Kiểm tra lại: import thành công 100% không còn lỗi!
!python -c "import timm; print('timm version:', timm.__version__, '-> Import OK!')"

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
timm version: 0.6.12 -> Import OK!


In [4]:
import os
from google.colab import userdata
import wandb
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login(key=os.environ["WANDB_API_KEY"])

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nguyennhatminhlk (nnm-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
!python download_data.py --benchmark all_6 --archive_dir ./data/archives

[INFO] Skipping 'cifar224': CIFAR-100 is automatically downloaded by torchvision during training and does not require a dataset.txt entry.
[DOWNLOAD] vtab -> ./data/archives/vtab.zip
Downloading...
From (original): https://drive.google.com/uc?id=1xUiwlnx4k0oDhYi26KL5KwrCAya-mvJ_
From (redirected): https://drive.google.com/uc?id=1xUiwlnx4k0oDhYi26KL5KwrCAya-mvJ_&confirm=t&uuid=6bc368e4-70ca-416e-a127-57b61a66c7a4
To: /content/LAMDA-PILOT/data/archives/vtab.zip.part
100% 127M/127M [00:01<00:00, 70.9MB/s]
[DOWNLOAD] cub -> ./data/archives/cub.zip
Downloading...
From (original): https://drive.google.com/uc?id=1XbUpnWpJPnItt5zQ6sHJnsjPncnNLvWb
From (redirected): https://drive.google.com/uc?id=1XbUpnWpJPnItt5zQ6sHJnsjPncnNLvWb&confirm=t&uuid=b7c48a08-adf6-4826-990f-a3a508e32eee
To: /content/LAMDA-PILOT/data/archives/cub.zip.part
100% 1.14G/1.14G [00:13<00:00, 84.9MB/s]
[DOWNLOAD] imageneta -> ./data/archives/imagenet-a.zip
Downloading...
From (original): https://drive.google.com/uc?id=19l52u

In [6]:
!python extract_data.py --dataset omnibenchmark
!python extract_data.py --dataset imagenetr

[EXTRACT] ./data/archives/omnibenchmark.zip -> ./data/omnibenchmark
[DONE] Extraction step complete.
[EXTRACT] ./data/archives/imagenet-r.zip -> ./data/imagenet-r
[DONE] Extraction step complete.


In [7]:
!python main.py --method simplecil --benchmark long_or_large --use_wandb --backbone_type pretrained_vit_b16_224_in21k


=== Running [1/2] preset=omnibench_b0inc30 dataset=omnibenchmark (b0 inc30) ===
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nguyennhatminhlk (nnm-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run bipi4tnh (0.3s)
wandb: ⣯ setting up run bipi4tnh (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/LAMDA-PILOT/wandb/run-20261121_041446-bipi4tnh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run PILOT_simplecil_omnibenchmark_b0inc30_s1993
wandb: ⭐️ View project at https://wandb.ai/nnm-hcmus/Group7-ML-Lab
wandb: 🚀 View run at https://wandb.ai/nnm-hcmus/Group7-ML-Lab/runs/bipi4tnh
2026-11-21 04:14:47

In [8]:
!python main.py --method aper_finetune --benchmark long_or_large --use_wandb --batch_size 72 --backbone_type pretrained_vit_b16_224_in21k


=== Running [1/2] preset=omnibench_b0inc30 dataset=omnibenchmark (b0 inc30) ===
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nguyennhatminhlk (nnm-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run mx6gxrko (0.3s)
wandb: ⣯ setting up run mx6gxrko (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/LAMDA-PILOT/wandb/run-20261121_044322-mx6gxrko
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run PILOT_aper_finetune_omnibenchmark_b0inc30_s1993
wandb: ⭐️ View project at https://wandb.ai/nnm-hcmus/Group7-ML-Lab
wandb: 🚀 View run at https://wandb.ai/nnm-hcmus/Group7-ML-Lab/runs/mx6gxrko
2026-11-21 04:4

In [9]:
!python main.py --method aper_ssf --benchmark long_or_large --use_wandb --backbone_type pretrained_vit_b16_224_in21k_ssf


=== Running [1/2] preset=omnibench_b0inc30 dataset=omnibenchmark (b0 inc30) ===
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nguyennhatminhlk (nnm-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run ekp0k0kl (0.3s)
wandb: ⣯ setting up run ekp0k0kl (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/LAMDA-PILOT/wandb/run-20261121_054800-ekp0k0kl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run PILOT_aper_ssf_omnibenchmark_b0inc30_s1993
wandb: ⭐️ View project at https://wandb.ai/nnm-hcmus/Group7-ML-Lab
wandb: 🚀 View run at https://wandb.ai/nnm-hcmus/Group7-ML-Lab/runs/ekp0k0kl
2026-11-21 05:48:01,

In [10]:
!python main.py --method aper_vpt_shallow --benchmark long_or_large --use_wandb --batch_size 96 --backbone_type pretrained_vit_b16_224_in21k_vpt


=== Running [1/2] preset=omnibench_b0inc30 dataset=omnibenchmark (b0 inc30) ===
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nguyennhatminhlk (nnm-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run qd4ngnxv (0.3s)
wandb: ⣯ setting up run qd4ngnxv (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/LAMDA-PILOT/wandb/run-20261121_073305-qd4ngnxv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run PILOT_aper_vpt_omnibenchmark_b0inc30_s1993
wandb: ⭐️ View project at https://wandb.ai/nnm-hcmus/Group7-ML-Lab
wandb: 🚀 View run at https://wandb.ai/nnm-hcmus/Group7-ML-Lab/runs/qd4ngnxv
2026-11-21 07:33:07,

In [ ]:
!python main.py --method aper_vpt_deep --benchmark long_or_large --use_wandb --batch_size 96 --backbone_type pretrained_vit_b16_224_in21k_vpt


=== Running [1/2] preset=omnibench_b0inc30 dataset=omnibenchmark (b0 inc30) ===
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nguyennhatminhlk (nnm-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 97mrm6lj (0.3s)
wandb: ⣯ setting up run 97mrm6lj (0.3s)
wandb: ⣟ setting up run 97mrm6lj (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/LAMDA-PILOT/wandb/run-20261121_091549-97mrm6lj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run PILOT_aper_vpt_omnibenchmark_b0inc30_s1993
wandb: ⭐️ View project at https://wandb.ai/nnm-hcmus/Group7-ML-Lab
wandb: 🚀 View run at https://wandb.ai/nnm-hcmus/Group7-M

In [7]:
!python main.py --method aper_adapter --benchmark long_or_large --use_wandb --batch_size 64 --backbone_type pretrained_vit_b16_224_in21k_adapter


=== Running [1/2] preset=omnibench_b0inc30 dataset=omnibenchmark (b0 inc30) ===
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nguyennhatminhlk (nnm-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 0os1k85c (0.3s)
wandb: ⣯ setting up run 0os1k85c (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/LAMDA-PILOT/wandb/run-20261121_131107-0os1k85c
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run PILOT_aper_adapter_omnibenchmark_b0inc30_s1993
wandb: ⭐️ View project at https://wandb.ai/nnm-hcmus/Group7-ML-Lab
wandb: 🚀 View run at https://wandb.ai/nnm-hcmus/Group7-ML-Lab/runs/0os1k85c
2026-11-21 13:11

In [8]:
from google.colab import runtime
runtime.unassign()